# Model Training Framework 2 
#### Code for training three model architectures. Compared to model training framework 1 the model inputs are normalized (inputs are between 0 and 1).

In [1]:
# Import necessary modules
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import os
from os import listdir
from os.path import isfile, join
from PIL import Image
import keras
from keras import layers
import csv
from tensorflow.keras.utils import to_categorical

2025-10-30 19:50:42.645764: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761853842.843421      37 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761853842.896844      37 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
# Read the data set 
eda = pd.read_csv('train_labels.csv')
eda

,id,label
0,f38a6374c348f90b587e046aac6079959adf3835,0
1,c18f2d887b7ae4f6742ee445113fa1aef383ed77,1
2,755db6279dae599ebb4d39a9123cce439965282d,0
3,bc3f0c64fb968ff4a8bd33af6971ecae77c75e08,0
4,068aba587a4950175d04c680d38943fd488d6a9d,0
...,...,...
220020,53e9aa9d46e720bf3c6a7528d1fca3ba6e2e49f6,0
220021,d4b854fe38b07fe2831ad73892b3cec877689576,1
220022,3d046cead1a2a5cbe00b2b4847cfb7ba7cf5fe75,0
220023,f129691c13433f66e1e0671ff1fe80944816f5a2,0


In [3]:
# Function to select a train set sample, because the 0/1 division is 60/40
# the sample is stratified this way
def sub_sample(frac_):
    zero_sample = int(len(eda)*frac_*0.6)
    eda_zero = eda[eda.label == 0]
    sample_eda_zero = eda_zero.sample(n = zero_sample, replace=False, random_state=52)

    # A sub sample of only one labels  
    one_sample = int(len(eda)*frac_*0.4)
    eda_one = eda[eda.label == 1]
    sample_eda_one = eda_one.sample(n = one_sample, replace=False, random_state=52)

    # Merge the sample data sets 
    frames = [sample_eda_zero, sample_eda_one]
    train_df = pd.concat(frames)

    # Shuffle the observations in random order
    train_df = train_df.sample(frac=1, random_state = 126)
    
    return train_df

In [4]:
# Select 10% of observations
train_df = sub_sample(0.1)
train_df

,id,label
176288,1d122a55476973f3847a705c96817c1e1b5ede9d,0
155346,5d94c8b0f0645c894c0ffcb292bbceb70393ea20,0
48863,d505c8b4fc7d65596ff3a8e6197b3a2474b68371,0
175558,11f7a527b4cabfcc4d00b455c9ecb52fc0547feb,1
195137,36ddfecad2e2b330e7bc8ba8fde52ddd2300168e,0
...,...,...
177151,db01a27193d200bc6d626f22d40fa2a441ebe082,1
211211,b3ed317e4997ad9435d2a8f9aeddcdb16fff079f,0
28330,f2552d8e74f0c4ccead7af168b9c3d5e2ce94bce,0
5173,112e7f5aff9bb19cfe15e8646ba6a5bcf7dbe4fd,0


In [5]:
# Function to transform .tif file to an array and to one hot encode the label 
def to_array(train_df = train_df):

    # String of working directory
    train_dir = '/kaggle/input/histopathologic-cancer-detection/train'
    
    imarray_totaal = []
    # Loop to import training images n train_df (by file names) from working directory
    for j in train_df.id:
    
        im = Image.open(train_dir + '/' + j + '.tif')
        # Turn '.tif' file into array
        imarray = np.array(im)
        
        imarray_totaal.append(imarray/255) # Normalize RGB values 
    
    # Turn list into array
    imarray_totaal= np.array(imarray_totaal)
    
    # One hot encoding for the label: Turning one label (0/1) into two labels (Label_0 and Label_1) 
    num_classes = 2
    labels = to_categorical(train_df.label, num_classes=num_classes)
    
    # Giving imarray_totaal (x_train) and labels (y_train) familiar names 
    x_train = imarray_totaal
    y_train = labels

    return x_train, y_train

In [6]:
# Function to write model training results to an external location
def write_away(name, model_results, path):

# Writing away the results 
    name = pd.DataFrame.from_dict(model_results)
    name.to_csv(path, index=False)
    return name

In [7]:
# Apply to_array
train_set = to_array()
x_train = train_set[0]
y_train = train_set[1]

### Model 1

In [8]:
num_classes = 2
input_shape = (96, 96, 3)

model = keras.Sequential(
    [keras.Input(shape = input_shape),
     layers.Conv2D(filters = 32, kernel_size=(3,3),activation = 'sigmoid'),
     layers.MaxPooling2D(pool_size=(2, 2)),
     layers.Flatten(),
     layers.Dense(num_classes, activation="softmax"),
    ]
    )
model.summary()

I0000 00:00:1761854331.221142      37 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1761854331.221891      37 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 94, 94, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 47, 47, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 70688)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │       141,378 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 142,274 (555.76 KB)

 Trainable params: 142,274 (555.76 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
# training model 1
batch_size = 200
epochs = 20
model.compile(loss="categorical_crossentropy", optimizer="sgd", metrics=["accuracy", "auc"])

history = model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_split=0.1)

Epoch 1/20


I0000 00:00:1761854371.312641      99 service.cc:148] XLA service 0x7b41b8005cb0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1761854371.313371      99 service.cc:156]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1761854371.313389      99 service.cc:156]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1761854371.470622      99 cuda_dnn.cc:529] Loaded cuDNN version 90300


  7/100 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - accuracy: 0.5473 - auc: 0.5243 - loss: 22.7486

I0000 00:00:1761854373.870396      99 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


100/100 ━━━━━━━━━━━━━━━━━━━━ 10s 63ms/step - accuracy: 0.5290 - auc: 0.5287 - loss: 15.8063 - val_accuracy: 0.3975 - val_auc: 0.3975 - val_loss: 12.3979
Epoch 2/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - accuracy: 0.5209 - auc: 0.5178 - loss: 2.5344 - val_accuracy: 0.6025 - val_auc: 0.5435 - val_loss: 3.9997
Epoch 3/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - accuracy: 0.5277 - auc: 0.5500 - loss: 1.3678 - val_accuracy: 0.6025 - val_auc: 0.5935 - val_loss: 1.4389
Epoch 4/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.5232 - auc: 0.5462 - loss: 1.0026 - val_accuracy: 0.6025 - val_auc: 0.5949 - val_loss: 1.5013
Epoch 5/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - accuracy: 0.5223 - auc: 0.5453 - loss: 0.9193 - val_accuracy: 0.3975 - val_auc: 0.4237 - val_loss: 3.1701
Epoch 6/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s 28ms/step - accuracy: 0.5405 - auc: 0.5635 - loss: 0.8699 - val_accuracy: 0.3975 - val_auc: 0.4110 - val_loss: 1.8026
Epoch 7/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 3s

In [10]:
# Model 1: Writing away the results 
write_away('Simple_CNN_Stan', history.history, 'Simple_CNN_Stan.csv')

,accuracy,auc,loss,val_accuracy,val_auc,val_loss
0,0.524418,0.530463,8.875331,0.397547,0.397547,12.397926
1,0.519721,0.528070,1.900904,0.602453,0.543502,3.999741
2,0.516893,0.544362,1.204802,0.602453,0.593473,1.438915
3,0.516338,0.545779,0.956163,0.602453,0.594887,1.501327
4,0.517651,0.544381,0.871332,0.397547,0.423664,3.170102
5,0.542902,0.570086,0.769257,0.397547,0.411044,1.802641
6,0.547094,0.580897,0.730675,0.397547,0.453697,7.182961
7,0.581688,0.617823,0.780262,0.398455,0.446458,0.901014
8,0.581082,0.624612,0.673556,0.602453,0.592189,1.815919
9,0.593859,0.642349,0.671295,0.602453,0.602316,1.263080


### Model 2

In [11]:
num_classes = 2
input_shape = (96, 96, 3)

model1 = keras.Sequential(
    [keras.Input(shape = input_shape),
     layers.Conv2D(filters = 32, kernel_size=(3,3),activation = 'sigmoid'),
     layers.MaxPooling2D(pool_size=(2, 2)),
     layers.Conv2D(64, kernel_size=(3, 3), activation="sigmoid"),
     layers.MaxPooling2D(pool_size=(2, 2)),
     layers.Flatten(),
     layers.Dense(num_classes, activation="softmax"),
    ]
    )
model1.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_1 (Conv2D)               │ (None, 94, 94, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 47, 47, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 45, 45, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 22, 22, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 30976)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │        61,954 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 81,346 (317.76 KB)

 Trainable params: 81,346 (317.76 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
# training model 2
batch_size = 200
epochs = 20

model1.compile(loss="categorical_crossentropy", optimizer="sgd", metrics=["accuracy", "auc"])

history1 = model1.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_split=0.1)

Epoch 1/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 12s 82ms/step - accuracy: 0.5129 - auc: 0.5277 - loss: 5.6841 - val_accuracy: 0.6025 - val_auc: 0.5687 - val_loss: 1.3243
Epoch 2/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 4s 44ms/step - accuracy: 0.5679 - auc: 0.5829 - loss: 0.7311 - val_accuracy: 0.6025 - val_auc: 0.5684 - val_loss: 1.5074
Epoch 3/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 4s 45ms/step - accuracy: 0.5644 - auc: 0.5878 - loss: 0.7292 - val_accuracy: 0.3975 - val_auc: 0.4336 - val_loss: 2.2572
Epoch 4/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 4s 45ms/step - accuracy: 0.5811 - auc: 0.5770 - loss: 0.7628 - val_accuracy: 0.6025 - val_auc: 0.5667 - val_loss: 1.1138
Epoch 5/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 45ms/step - accuracy: 0.5968 - auc: 0.6013 - loss: 0.6962 - val_accuracy: 0.6025 - val_auc: 0.5690 - val_loss: 1.0767
Epoch 6/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 45ms/step - accuracy: 0.5995 - auc: 0.6023 - loss: 0.6936 - val_accuracy: 0.6025 - val_auc: 0.5685 - val_loss: 1.0363
Epoch 7/20
100/100 ━━━━━━━━━━━━━━

In [13]:
# Model 2: Writing away the results 
write_away('Basic_CNN_Stan', history1.history, 'Basic_CNN_Stan.csv')

,accuracy,auc,loss,val_accuracy,val_auc,val_loss
0,0.502954,0.526023,2.790148,0.602453,0.568744,1.324263
1,0.576133,0.590801,0.694474,0.602453,0.568394,1.507358
2,0.574466,0.593498,0.692543,0.397547,0.433592,2.257185
3,0.590526,0.592526,0.695810,0.602453,0.566668,1.113800
4,0.591485,0.598306,0.682937,0.602453,0.569040,1.076746
5,0.596233,0.600409,0.681507,0.602453,0.568504,1.036346
6,0.597091,0.599663,0.680184,0.602453,0.568249,1.096869
7,0.599919,0.604120,0.680231,0.397547,0.435060,1.639474
8,0.592950,0.597627,0.686293,0.397547,0.434901,1.597639
9,0.595475,0.599838,0.684119,0.397547,0.434480,1.459149


In [14]:
# Select 20% of observations
train_df = sub_sample(0.2)
train_df

,id,label
101866,8312c9bac5ab0dded0b79b0e8793ac4470727f40,0
104382,146db8073ee4fc6e64dac8cc8b835306ce4f00a5,1
123473,d6bfa926359cdffe8a770c4c6513322924825928,1
201216,9e2bb84236b7adcd4d245dd6ac9d573bea10204b,0
62800,1809061f44efe7f494c72da733ba50f6a5f054c9,0
...,...,...
110944,18b62ca10f10a13b9dcab6c377a69e3afbb4f716,0
118348,74e880c6deb43c4d0a31adba76becb1eebfaa813,1
28330,f2552d8e74f0c4ccead7af168b9c3d5e2ce94bce,0
5173,112e7f5aff9bb19cfe15e8646ba6a5bcf7dbe4fd,0


In [15]:
# Apply function to_array()
train_set = to_array()
x_train = train_set[0]
y_train = train_set[1]

### Model 3 

In [16]:
num_classes = 2
input_shape = (96, 96, 3)

model2 = keras.Sequential(
    [keras.Input(shape = input_shape),
     layers.Conv2D(filters = 32, kernel_size=(3,3),activation = 'sigmoid'),
     layers.MaxPooling2D(pool_size=(2, 2)),
     layers.Conv2D(64, kernel_size=(3, 3), activation="sigmoid"),
     layers.MaxPooling2D(pool_size=(2, 2)),
     layers.Flatten(),
     layers.Dense(num_classes, activation="softmax"),
    ]
    )
model2.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 94, 94, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 47, 47, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 45, 45, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 22, 22, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 30976)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │        61,954 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 81,346 (317.76 KB)

 Trainable params: 81,346 (317.76 KB)

 Non-trainable params: 0 (0.00 B)

In [17]:
# training model 3
batch_size = 200
epochs = 20

model2.compile(loss="categorical_crossentropy", optimizer="sgd", metrics=["accuracy", "auc"])

history2 = model2.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_split=0.1)

Epoch 1/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 10s 77ms/step - accuracy: 0.5151 - auc: 0.5309 - loss: 5.5644 - val_accuracy: 0.6025 - val_auc: 0.5943 - val_loss: 1.7875
Epoch 2/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 47ms/step - accuracy: 0.5821 - auc: 0.5938 - loss: 0.7426 - val_accuracy: 0.3975 - val_auc: 0.4348 - val_loss: 2.1529
Epoch 3/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 47ms/step - accuracy: 0.5767 - auc: 0.5831 - loss: 0.7540 - val_accuracy: 0.6025 - val_auc: 0.5640 - val_loss: 1.2362
Epoch 4/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 47ms/step - accuracy: 0.5923 - auc: 0.5979 - loss: 0.7073 - val_accuracy: 0.6025 - val_auc: 0.5635 - val_loss: 1.1376
Epoch 5/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 47ms/step - accuracy: 0.5945 - auc: 0.5943 - loss: 0.7053 - val_accuracy: 0.6025 - val_auc: 0.5646 - val_loss: 1.1273
Epoch 6/20
100/100 ━━━━━━━━━━━━━━━━━━━━ 5s 47ms/step - accuracy: 0.5940 - auc: 0.5998 - loss: 0.7030 - val_accuracy: 0.6025 - val_auc: 0.5648 - val_loss: 1.0690
Epoch 7/20
100/100 ━━━━━━━━━━━━━━

In [18]:
# Model 3: Writing away the results 
write_away('Basic_CNN_Stan_20per', history2.history, 'Basic_CNN_Stan_20per.csv')

,accuracy,auc,loss,val_accuracy,val_auc,val_loss
0,0.519772,0.535073,2.749236,0.602453,0.594298,1.787450
1,0.567800,0.580332,0.709732,0.397547,0.434752,2.152918
2,0.582546,0.592261,0.696708,0.602453,0.564022,1.236186
3,0.592344,0.597855,0.687145,0.602453,0.563457,1.137594
4,0.592596,0.594955,0.685304,0.602453,0.564553,1.127322
5,0.597394,0.600792,0.681985,0.602453,0.564787,1.068975
6,0.594515,0.599110,0.682145,0.397547,0.436269,1.618063
7,0.595374,0.599326,0.684660,0.602453,0.564976,1.071556
8,0.595980,0.601061,0.680443,0.397547,0.439769,1.562626
9,0.596384,0.603712,0.683083,0.602453,0.565678,1.034817
